In [1]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("all-MiniLM-L6-v2")
print("Model loaded")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Model loaded


In [2]:
q1 = "Can I still join the course after the start date?"
q2 = "How to install Docker on Windows?"
doc = "You don't need to register. You're accepted. You can also just start learning and submitting homework without registering."


In [4]:
v1 = model.encode(q1)
v2 = model.encode(q2)
dv = model.encode(doc)
print(f"Vector shape: {v1.shape}")

Vector shape: (384,)


In [13]:
print(f"q1 vs doc: {v1.dot(dv)}")   # should be ~0.32 - related
print(f"q2 vs doc: {v2.dot(dv)}")   # should be ~0.01 - unrelated

q1 vs doc: 0.3233239948749542
q2 vs doc: 0.01973051019012928


In [6]:
from ingest import load_faq_data

documents = load_faq_data()
print(f"Total documents: {len(documents)}")
print(f"Sample doc: {documents[0]}")

Total documents: 1350
Sample doc: {'course': 'data-engineering-zoomcamp', 'section': 'General Course-Related Questions', 'question': 'Course: When does the course start?', 'answer': "A new cohort runs roughly January–April every year. For the current cohort's exact start date and registration link, check the [course repo README](https://github.com/DataTalksClub/data-engineering-zoomcamp).\n\n- Register via the link in the course repo before the cohort starts.\n- Join the [course Telegram channel](https://t.me/dezoomcamp) for announcements.\n- Join DataTalks.Club's [Slack](https://datatalks.club/docs/general/slack/) and the `#course-data-engineering` channel.", 'doc_id': '9e508f2212'}


In [7]:
texts = []

for doc in documents:
    text = doc["question"] + " " + doc["answer"]
    texts.append(text)

print(f"Total texts: {len(texts)}")
print(f"Sample text: {texts[0][:100]}...")

Total texts: 1350
Sample text: Course: When does the course start? A new cohort runs roughly January–April every year. For the curr...


In [8]:
from tqdm.auto import tqdm
import numpy as np

batch_size = 50
vectors = []

for i in tqdm(range(0, len(texts), batch_size)):
    batch = texts[i:i + batch_size]
    batch_vectors = model.encode(batch)
    vectors.extend(batch_vectors)

X = np.array(vectors)
print(f"Matrix shape: {X.shape}")

  0%|          | 0/27 [00:00<?, ?it/s]

Matrix shape: (1350, 384)


In [11]:
query = "Can I still join the course after the start date?"
v_query = model.encode(query)

scores = X.dot(v_query)

print(f"Query vector shape: {v_query.shape}")
print(f"Scores shape: {scores.shape}")
print(f"Highest score: {scores.max()}")
print(f"Lowest score:  {scores.min()}")

Query vector shape: (384,)
Scores shape: (1350,)
Highest score: 0.7629411220550537
Lowest score:  -0.1448725312948227


In [12]:
idx = np.argmax(scores)

print(f"Best match index: {idx}")
print(f"Best match score: {scores[idx]}")
print(f"Document: {documents[idx]}")

Best match index: 2
Best match score: 0.7629411220550537
Document: {'course': 'data-engineering-zoomcamp', 'section': 'General Course-Related Questions', 'question': 'Course: Can I still join the course after the start date?', 'answer': "Yes, even if you don't register, you're still eligible to submit the homework.\n\nBe aware, however, that there will be deadlines for turning in homeworks and the final projects. So don't leave everything for the last minute.", 'doc_id': '3f1424af17'}


In [16]:
top5 = np.argsort(-scores)[:5]

for idx in top5:
    print(f"Score: {scores[idx]:.4f}")
    print(f"Q: {documents[idx]['question']}")
    print(f"A: {documents[idx]['answer'][:80]}...")
    print()

Score: 0.7629
Q: Course: Can I still join the course after the start date?
A: Yes, even if you don't register, you're still eligible to submit the homework.

...

Score: 0.7579
Q: Course - Can I still join the course after the start date?
A: Yes, even if you don't register, you're still eligible to submit the homeworks a...

Score: 0.7192
Q: The course has already started. Can I still join it?
A: Yes, you can. Even though you missed the start date, you can register for the co...

Score: 0.6536
Q: I just discovered the course. Can I still join?
A: Yes, but if you want to receive a certificate, you need to submit your project w...

Score: 0.5601
Q: Course - Can I follow the course after it finishes?
A: Yes, we will keep all the materials available, so you can follow the course at y...



In [17]:
query2 = "How do I install Docker on Windows?"
v_query2 = model.encode(query2)
scores2 = X.dot(v_query2)
top5_q2 = np.argsort(-scores2)[:5]

for idx in top5_q2:
    print(f"Score: {scores2[idx]:.4f}")
    print(f"Q: {documents[idx]['question']}")
    print()

Score: 0.6345
Q: Docker: Docker not installable on Ubuntu

Score: 0.6040
Q: Install docker in WSL2 without installing Docker Desktop

Score: 0.5819
Q: Install docker on MacOS

Score: 0.5782
Q: WSL: instructions

Score: 0.5582
Q: Docker: Cannot connect to the docker daemon. Is the Docker daemon running?



In [18]:
from minsearch import VectorSearch

vindex = VectorSearch(keyword_fields=["course"])
vindex.fit(X, documents)

print("Index built")

Index built


In [19]:
query = "I just discovered the course. Can I still join it?"
query_vector = model.encode(query)

results = vindex.search(query_vector, num_results=5)

results[0]

{'course': 'llm-zoomcamp',
 'section': 'General Course-Related Questions',
 'question': 'I just discovered the course. Can I still join?',
 'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.',
 'doc_id': '74eb249bbf'}

In [20]:
results_filtered = vindex.search(
    query_vector,
    filter_dict={"course": "llm-zoomcamp"},
    num_results=5
)

print("Filtered to llm-zoomcamp only:\n")
for r in results_filtered:
    print(f"[{r['course']}] {r['question']}")
    print()

Filtered to llm-zoomcamp only:

[llm-zoomcamp] I just discovered the course. Can I still join?

[llm-zoomcamp] Certificate: Can I follow the course in a self-paced mode and get a certificate?

[llm-zoomcamp] When will the course be offered next?

[llm-zoomcamp] Can I run the course locally instead of Codespaces?

[llm-zoomcamp] OpenAI: Do I have to subscribe and pay for Open AI API for this course?



In [21]:
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
openai_client = OpenAI()


In [22]:
from ingest import load_faq_data, build_index

documents = load_faq_data()
index = build_index(documents)

In [23]:
from rag_helper import RAGBase

assistant = RAGBase(
    index=index,
    llm_client=openai_client,
)

In [24]:
query = "I just found out about the program, can I still sign up?"
assistant.rag(query)

'Yes, but if you want to receive a certificate, you need to submit your project while they’re still accepting submissions.'

In [25]:
from rag_helper import RAGBase

class RAGVector(RAGBase):

    def __init__(self, embedder, **kwargs):
        super().__init__(**kwargs)
        self.embedder = embedder

    def search(self, query, num_results=5):
        query_vector = self.embedder.encode(query)
        filter_dict = {"course": self.course}

        return self.index.search(
            query_vector,
            num_results=num_results,
            filter_dict=filter_dict
        )

In [26]:
vector_assistant = RAGVector(
    embedder=model,
    index=vindex,
    llm_client=openai_client,
)


In [27]:
vector_assistant.rag("the program has already incepted, can I still sign up?")

'Yes, you can still join. If you want a certificate, make sure to submit your project while submissions are still open.'

In [28]:
# Paraphrase of a Docker question
answer2 = vector_assistant.rag("do I need to containerize my solution?")
print(answer2)
print("---")

# Something vague
answer3 = vector_assistant.rag("what happens if I miss the deadline?")
print(answer3)

I don't know.
---
I don’t know.
